# Imports

In [4]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load data

In [5]:
maison_df = pd.read_csv('datasets_prepd/dvf_maison.csv')
print(maison_df.head())

    id_mutation date_mutation  numero_disposition nature_mutation  \
0  2021-1289503    2021-01-05                   1           Vente   
1  2021-1289509    2021-01-06                   1           Vente   
2  2021-1289511    2021-01-04                   1           Vente   
3  2021-1289514    2021-01-05                   1           Vente   
4  2021-1289517    2021-01-04                   1           Vente   

   valeur_fonciere  adresse_numero adresse_suffixe      adresse_nom_voie  \
0         352000.0           228.0             NaN       RUE DE L EGLISE   
1         334800.0             7.0             NaN          AV ROSCOMMON   
2         225700.0            35.0             NaN  RUE DE LA REPUBLIQUE   
3         224000.0             7.0             NaN     CHE DE LA GARENNE   
4         267000.0            11.0             NaN     ALL DES AUBEPINES   

  adresse_code_voie  code_postal  ...  nature_culture_speciale  \
0              0220      77115.0  ...                      NaN

# Features

In [ ]:
features = [
    "longitude",
    "latitude",
    "code_postal",
    "surface_reelle_bati",
    "nombre_pieces_principales",
    "prix_m2_ref",
    "surface_terrain",
    "number_of_lots",
    "season"
]

target = "valeur_fonciere_actualisee"

# Encode season to numeric values
season_mapping = {"winter": 0, "spring": 1, "summer": 2, "autumn": 3}
maison_df["season"] = maison_df["season"].map(season_mapping)

X = maison_df[features]
y = maison_df[target]

print("Number of features:", X.shape[1])
print(X.head())
print(y.head())

KeyError: "['total_surface_terrain'] not in index"

# Make sets : trail and test and validate

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

# Ssave

In [ ]:
os.makedirs("data_maison", exist_ok=True)

train_df = X_train.copy()
val_df = X_val.copy()
test_df = X_test.copy()

train_df[target] = y_train
val_df[target] = y_val
test_df[target] = y_test

train_df.to_csv("data_maison/maison_train.csv", index=False)
val_df.to_csv("data_maison/maison_val.csv", index=False)
test_df.to_csv("data_maison/maison_test.csv", index=False)

print("Datasets saved.")

Train model

In [ ]:
model_maison = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)

model_maison.fit(X_train, y_train)

print("Model trained.")

Evaluation

In [ ]:
y_val_pred = model_maison.predict(X_val)

rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
mae = mean_absolute_error(y_val, y_val_pred)
r2 = r2_score(y_val, y_val_pred)

print("Validation RMSE:", rmse)
print("Validation MAE:", mae)
print("Validation R2:", r2)

# Test

In [ ]:
y_test_pred = model_maison.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
mae = mean_absolute_error(y_test, y_test_pred)
r2 = r2_score(y_test, y_test_pred)

print("Test RMSE:", rmse)
print("Test MAE:", mae)
print("Test R2:", r2)

Save model

In [ ]:
os.makedirs("models", exist_ok=True)

joblib.dump(model_maison, "models/maison_random_forest_model.pkl")

print("Model saved.")

In [ ]:
print("Number of features expected:", model_maison.n_features_in_)
print("Feature names:", features)